# SBI 811 — Lecture 7

# Reading, Writing and Processing FASTA Files

## Computer Programming in Bioinformatics

Programming becomes especially useful when information is stored in files. A program can read hundreds or thousands of records, check their contents, calculate summaries and write new results without opening every file manually.

This lecture begins with ordinary text files and then applies the same ideas to FASTA files. FASTA is a widely used plain-text format for storing biological sequences.

The examples use only Python features already introduced in SBI 811:

- strings;
- lists;
- loops;
- `if`, `elif` and `else` statements;
- functions;
- dictionaries; and
- simple debugging and testing.

No pandas knowledge is required.

# Part 1 — Information in memory and information in files

When Python runs this instruction:

```python
sequence = "ATGCGT"
```

the string is stored in the computer's working memory. The variable normally disappears when the Python session ends.

A **file** stores information on a storage device. The information can remain available after Python stops. A later program can open the same file and read it again.

This distinction is important:

| Location | Example | Lifetime |
|---|---|---|
| Python memory | `sequence = "ATGCGT"` | Usually until the session ends |
| File storage | `sequences.fasta` | Remains until changed or deleted |

Reading a file transfers information from storage into Python. Writing a file transfers information from Python to storage.

# Part 2 — Understanding file paths

A **file path** describes the location of a file.

An absolute Linux path starts at the root directory `/`:

```text
/home/student/SBI811/lecture07_sequences.fasta
```

A relative path starts from the directory in which the notebook is running:

```text
lecture07_sequences.fasta
```

The notebook uses a relative path because the FASTA file should be saved in the same directory as the notebook.

Python can report its current working directory using the `Path` class from the `pathlib` module.

In [1]:
from pathlib import Path

# current_directory = Path.cwd()
# print("Current directory:", current_directory)

Path.cwd() #pwd

PosixPath('/home/cwekesa/2026/BioinfoClassRoom/SBI811')

### Explanation of the code

```python
from pathlib import Path
```

- `pathlib` is a Python module for working with paths.
- `Path` is one class contained in that module.
- `from ... import ...` makes the selected class available under its own name.

```python
current_directory = Path.cwd()
```

- `Path.cwd()` asks for the **current working directory**.
- The returned path is stored in `current_directory`.

```python
print("Current directory:", current_directory)
```

`print()` displays a label followed by the path.

## Check whether the supplied FASTA file is visible

Before opening a file, it is useful to check that Python can see it.

In [2]:
fasta_path = Path("lecture07_sequences.fasta")

print("File exists:", fasta_path.exists())
print("Full path:", fasta_path.resolve())

File exists: True
Full path: /home/cwekesa/2026/BioinfoClassRoom/SBI811/lecture07_sequences.fasta


`Path("lecture07_sequences.fasta")` creates a path object. It does not create or open the file.

`.exists()` returns a Boolean value:

- `True` means something exists at the path;
- `False` means Python cannot find it from the current location.

`.resolve()` displays an absolute form of the path. If `.exists()` returns `False`, check that the notebook and FASTA file are in the same directory and that the capitalization is identical.

Linux filenames are case-sensitive. These can be different files:

```text
sequences.fasta
Sequences.fasta
SEQUENCES.FASTA
```

# Part 3 — Opening and closing a text file

The built-in `open()` function connects Python to a file.

```python
file_object = open("lecture07_sequences.fasta", "r")
```

The first argument is the path. The second argument is the **mode**.

Common modes are:

| Mode | Meaning | Important behaviour |
|---|---|---|
| `"r"` | read | File must already exist |
| `"w"` | write | Creates a file or erases existing contents |
| `"a"` | append | Adds new text at the end |

After using a directly opened file, it should be closed:

```python
file_object.close()
```

Forgetting to close a file can delay writing, consume resources or leave the file in an uncertain state. Python therefore provides the `with` statement.

## The recommended form: `with open(...)`

In [4]:
with open(file = "raw_data/lecture07_sequences.fasta",mode='r') as f:
    print(f.read())

>seq01 example bacterial enzyme
ATGCGTACGTTAGCTAGCTAGGCTAACCGGTTAA
>seq02 example regulatory sequence
TTGACATGCGCGATATATCGCGGATCCGATTA
>seq03 sequence containing lowercase letters
atggccattgtaatgggccgctgaaagggtgcccgatag
>seq04 sequence split across several lines
ATGAAACCCGGGTTT
AAACCCGGGTTTAAA

CCCGGGTTTAAATAG
>seq05 sequence with an ambiguous nucleotide
ATGCGTNNNACGTACGTAG



In [2]:
with open("raw_data/lecture07_sequences.fasta", "r") as fasta_file: 
    file_contents = fasta_file.read()

print(file_contents)

>seq01 example bacterial enzyme
ATGCGTACGTTAGCTAGCTAGGCTAACCGGTTAA
>seq02 example regulatory sequence
TTGACATGCGCGATATATCGCGGATCCGATTA
>seq03 sequence containing lowercase letters
atggccattgtaatgggccgctgaaagggtgcccgatag
>seq04 sequence split across several lines
ATGAAACCCGGGTTT
AAACCCGGGTTTAAA

CCCGGGTTTAAATAG
>seq05 sequence with an ambiguous nucleotide
ATGCGTNNNACGTACGTAG



Read the first line in pieces:

```python
with open("lecture07_sequences.fasta", "r") as fasta_file:
```

- `open(...)` opens the named file in read mode.
- `as fasta_file` gives the opened file object a variable name.
- The colon `:` begins the indented `with` block.
- `fasta_file` represents the open connection, not the filename text.

Inside the block:

```python
file_contents = fasta_file.read()
```

`.read()` reads all remaining characters and returns one string. That string is stored in `file_contents`.

When the indented block finishes, Python closes the file automatically—even if an error occurs inside the block. The variable `file_contents` remains available because it contains the text already read into memory.

The name `fasta_file` may still exist after the block, but its file connection is closed and should no longer be read from.

# Part 4 — Three common reading methods

Python file objects provide several reading methods.

## 4.1 `.read()` — one string containing the remaining file

```python
with open("file.txt", "r") as file:
    text = file.read()
```

This is simple for a small file. A very large file may not fit comfortably in memory.

## 4.2 `.readline()` — one line

Each call reads the next line.

In [5]:
with open("raw_data/lecture07_sequences.fasta", "r") as fasta_file:
    first_line = fasta_file.readline()
    second_line = fasta_file.readline()
    third_line = fasta_file.readline()

print("First:", repr(first_line))
print("Second:", repr(second_line))
print("Third:", repr(third_line))

First: '>seq01 example bacterial enzyme\n'
Second: 'ATGCGTACGTTAGCTAGCTAGGCTAACCGGTTAA\n'
Third: '>seq02 example regulatory sequence\n'


`repr()` displays a representation of a string that exposes special characters. The output may contain `\n` at the end of each line.

`\n` is the newline character. It records where one text line ends and the next begins. It is one character even though its representation uses a backslash and the letter `n`.

The file position advances after every call to `.readline()`. The first call returns line 1, the second returns line 2 and so on. At the end of the file, `.readline()` returns an empty string `""`.

## 4.3 `.readlines()` — a list of lines

In [8]:
with open("raw_data/lecture07_sequences.fasta", "r") as fasta_file:
    lines = fasta_file.readlines()

print("Type:", type(lines))
print("Number of lines:", len(lines))
print("First list item:", repr(lines[0]))

# print(lines)

Type: <class 'list'>
Number of lines: 13
First list item: '>seq01 example bacterial enzyme\n'


`.readlines()` returns a list. Each list item is one line, usually including its newline character.

This allows indexing and slicing:

```python
lines[0]      # first line
lines[-1]     # final line
lines[:3]     # first three lines
```

Like `.read()`, `.readlines()` loads the remaining file contents into memory. For large files, looping directly over the file is more memory-efficient.

# Part 5 — Reading a file one line at a time

An open text file is iterable. A `for` loop can request one line at a time.

In [13]:
with open("raw_data/lecture07_sequences.fasta", "r") as fasta_file:
    for line in fasta_file:
        #print(repr(line))
        print(line)

>seq01 example bacterial enzyme

ATGCGTACGTTAGCTAGCTAGGCTAACCGGTTAA

>seq02 example regulatory sequence

TTGACATGCGCGATATATCGCGGATCCGATTA

>seq03 sequence containing lowercase letters

atggccattgtaatgggccgctgaaagggtgcccgatag

>seq04 sequence split across several lines

ATGAAACCCGGGTTT

AAACCCGGGTTTAAA



CCCGGGTTTAAATAG

>seq05 sequence with an ambiguous nucleotide

ATGCGTNNNACGTACGTAG



The loop performs these steps repeatedly:

1. obtain the next line from `fasta_file`;
2. store that line temporarily in `line`;
3. execute the indented body;
4. stop automatically when no lines remain.

Only one line needs to be held for the loop operation at a time. This pattern is preferred for large sequence files.

The name `line` is not special. A descriptive alternative could be `fasta_line`. The chosen variable name should help the reader understand what one loop item represents.

# Part 6 — Removing newline characters

The newline is part of the string returned from a text file. It is often removed before analysing a line.

## `.strip()`

`.strip()` removes whitespace from both ends of a string. Whitespace includes spaces, tabs and newline characters.

## `.rstrip()`

`.rstrip()` removes whitespace only from the right-hand end.

## `.lstrip()`

`.lstrip()` removes whitespace only from the left-hand end.

In [14]:
example_line = "  ATGCGT  \n"

print("Original:", repr(example_line))
print("strip():", repr(example_line.strip()))
print("rstrip():", repr(example_line.rstrip()))
print("lstrip():", repr(example_line.lstrip()))

Original: '  ATGCGT  \n'
strip(): 'ATGCGT'
rstrip(): '  ATGCGT'
lstrip(): 'ATGCGT  \n'


For clean FASTA sequence lines, `.strip()` is commonly acceptable because surrounding whitespace is not part of the biological sequence.

For some other text formats, leading spaces may carry meaning. In that case, `.rstrip("\n")` can remove only the newline:

```python
line_without_newline = line.rstrip("\n")
```

This distinction matters:

- `.rstrip()` with no argument removes all types of trailing whitespace;
- `.rstrip("\n")` removes newline characters from the right;
- `.strip()` removes whitespace from both ends.

A method returns a new string. Strings are immutable, so the original string is not modified unless the returned value is assigned.

In [11]:
raw_line = "ATGCGT\n"
raw_line.strip()

print("Without assignment:", repr(raw_line))

clean_line = raw_line.strip()
print("Stored result:", repr(clean_line))

Without assignment: 'ATGCGT\n'
Stored result: 'ATGCGT'


Calling `raw_line.strip()` calculates a cleaned string, but the result is lost if it is not stored or used. The assignment to `clean_line` preserves the returned string.

# Part 7 — The FASTA format

A FASTA file stores one or more sequence records as plain text.

A record contains:

1. a header line beginning with `>`;
2. one or more sequence lines.

Example:

```text
>seq01 example sequence
ATGCGTACGTTAGC
```

The greater-than symbol `>` identifies a header. The text immediately following it often contains a sequence identifier. Additional description may follow.

A multi-record FASTA file can look like:

```text
>seq01 first sequence
ATGCGT
AACCGG
>seq02 second sequence
TTGCAA
```

Important observations:

- a sequence may occupy one line or several lines;
- a new record begins when another header is encountered;
- blank lines are best ignored or treated according to a defined rule;
- nucleotide sequences may use uppercase or lowercase letters;
- ambiguous symbols such as `N` may occur;
- the FASTA format itself does not guarantee that sequence characters are valid.

A parser must not assume that one sequence always occupies exactly one line.

# Part 8 — Identifying headers and sequence lines

The string method `.startswith()` checks the beginning of a string.

In [22]:
with open("raw_data/lecture07_sequences.fasta", "r") as fasta_file:
    for line in fasta_file:
        clean_line = line.strip()
        
        if clean_line.startswith(">"):
            print("HEADER:", clean_line)
        else:
            print("SEQUENCE:", clean_line)

HEADER: >seq01 example bacterial enzyme
SEQUENCE: ATGCGTACGTTAGCTAGCTAGGCTAACCGGTTAA
HEADER: >seq02 example regulatory sequence
SEQUENCE: TTGACATGCGCGATATATCGCGGATCCGATTA
HEADER: >seq03 sequence containing lowercase letters
SEQUENCE: atggccattgtaatgggccgctgaaagggtgcccgatag
HEADER: >seq04 sequence split across several lines
SEQUENCE: ATGAAACCCGGGTTT
SEQUENCE: AAACCCGGGTTTAAA
SEQUENCE: 
SEQUENCE: CCCGGGTTTAAATAG
HEADER: >seq05 sequence with an ambiguous nucleotide
SEQUENCE: ATGCGTNNNACGTACGTAG


In [19]:
name = "John Doe"
name.endswith('e')

True

The condition:

```python
clean_line.startswith(">")
```

returns `True` when the first character is `>` and `False` otherwise.

The `if` block runs for headers. The `else` block runs for every line that is not a header. This first version would label a blank line as a sequence line. A more careful version tests for blank lines explicitly.

In [24]:
with open("raw_data/lecture07_sequences.fasta", "r") as fasta_file:
    for line in fasta_file:
        clean_line = line.strip()

        if clean_line == "":
            continue
        elif clean_line.startswith(">"):
            print("HEADER:", clean_line)
        else:
            print("SEQUENCE:", clean_line)

HEADER: >seq01 example bacterial enzyme
SEQUENCE: ATGCGTACGTTAGCTAGCTAGGCTAACCGGTTAA
HEADER: >seq02 example regulatory sequence
SEQUENCE: TTGACATGCGCGATATATCGCGGATCCGATTA
HEADER: >seq03 sequence containing lowercase letters
SEQUENCE: atggccattgtaatgggccgctgaaagggtgcccgatag
HEADER: >seq04 sequence split across several lines
SEQUENCE: ATGAAACCCGGGTTT
SEQUENCE: AAACCCGGGTTTAAA
SEQUENCE: CCCGGGTTTAAATAG
HEADER: >seq05 sequence with an ambiguous nucleotide
SEQUENCE: ATGCGTNNNACGTACGTAG


`clean_line == ""` asks whether the cleaned line is an empty string.

`continue` immediately moves to the next loop iteration. No later code in the current iteration is executed.

The order of the conditions is deliberate:

1. ignore blank lines;
2. identify headers;
3. treat remaining non-empty lines as sequence lines.

This still does not prove that a remaining line contains a valid sequence. Validation is added later.

# Part 9 — Reading a single FASTA record

Begin with the simpler case of one header and one or more sequence lines.

The goal is to produce:

```python
identifier = "seq01"
sequence = "ATGCGTAACCGG"
```

Sequence fragments are collected in a list and joined after reading.

In [39]:
sequence = "ATGCGTAACCGG"

# startswith
if sequence.startswith('A'):
    print('yes')
    
# endswith
if sequence.endswith('G'):
    print('It was G')
    
# split()
x = "We,are,learning,today"
x.split(',')

#join()
y = ['We', 'are', 'learning', 'today']
"".join(y)



yes
It was G
We
learning
today


In [42]:
#continue
y = ['We', 'are', 'learning', 'today']
for i in y:
    if i == 'are':
        continue
        
    print(i)    

We
learning
today


In [44]:
#break
y = ['We', 'are', 'learning', 'today']
for i in y:
    if i == 'are':
        break
        
    print(i)    

We


# Part 10 — Designing a multi-record FASTA parser

A multi-record parser must know when one record finishes and the next begins.

Useful state variables are:

- `current_id` — identifier of the record currently being collected;
- `current_parts` — list of sequence lines for that record;
- `sequences` — dictionary containing completed records.

When a new header appears:

1. save the previous record if one exists;
2. extract the new identifier;
3. reset the sequence-parts list.

After the loop, save the final record because no later header appears to trigger its storage.

## Pseudocode

```text
create an empty dictionary
set current identifier to nothing
create an empty list of sequence parts

open the FASTA file
for each line:
    remove surrounding whitespace
    if the line is blank:
        skip it
    if the line is a header:
        if a previous identifier exists:
            join and store its sequence parts
        extract the new identifier
        reset the sequence-parts list
    otherwise:
        add the line to the current sequence parts

after the loop:
    store the final record
```

In [3]:
sequences = {}
current_id = None
current_parts = []

with open("raw_data/lecture07_sequences.fasta", "r") as fasta_file:
    for line in fasta_file:
        clean_line = line.strip()

        if clean_line == "":
            continue

        if clean_line.startswith(">"):
            if current_id is not None:
                complete_sequence = "".join(current_parts)
                sequences[current_id] = complete_sequence

            header_text = clean_line[1:]
            current_id = header_text.split()[0]
            current_parts = []
        else:
            current_parts.append(clean_line)
            

    if current_id is not None:
        complete_sequence = "".join(current_parts)
        sequences[current_id] = complete_sequence

print(sequences)

{'seq01': 'ATGCGTACGTTAGCTAGCTAGGCTAACCGGTTAA', 'seq02': 'TTGACATGCGCGATATATCGCGGATCCGATTA', 'seq03': 'atggccattgtaatgggccgctgaaagggtgcccgatag', 'seq04': 'ATGAAACCCGGGTTTAAACCCGGGTTTAAACCCGGGTTTAAATAG', 'seq05': 'ATGCGTNNNACGTACGTAG'}


In [4]:
# {"seq01":"ATGCGTACGTTAGCTAGCTAGGCTAACCGGTTAA","seq04":"ATGAAACCCGGGTTTAAACCCGGGTTTAAACCCGGGTTTAAATAG"}

def fasta_reader(file):
    
    sequences = {}
    current_id = None
    current_parts = []

    with open(file, "r") as fasta_file:
        for line in fasta_file:
            line = line.strip()

            if line == "":
                continue

            if line.startswith('>'):
                if current_id is not None:
                    complete_sequence = "".join(current_parts)
                    sequences[current_id] = complete_sequence

                line_without_fasta_sign = line[1:]
                first_part_of_line = line_without_fasta_sign.split()[0]
                current_id = first_part_of_line
                current_parts = []

            else:
                current_parts.append(line)

        if current_id is not None:
            complete_sequence = "".join(current_parts)
            sequences[current_id] = complete_sequence


    return sequences        

fasta_reader("raw_data/lecture07_sequences.fasta")

{'seq01': 'ATGCGTACGTTAGCTAGCTAGGCTAACCGGTTAA',
 'seq02': 'TTGACATGCGCGATATATCGCGGATCCGATTA',
 'seq03': 'atggccattgtaatgggccgctgaaagggtgcccgatag',
 'seq04': 'ATGAAACCCGGGTTTAAACCCGGGTTTAAACCCGGGTTTAAATAG',
 'seq05': 'ATGCGTNNNACGTACGTAG'}

### Understanding the state variables

```python
sequences = {}
```

creates an empty dictionary. Identifiers will become keys and sequence strings will become values.

```python
current_id = None
```

`None` represents the absence of a current identifier before the first header is read. It is different from the text string `"None"`.

```python
current_parts = []
```

creates an empty list for the current record's sequence lines.

### Why test `current_id is not None`?

When the first header appears, there is no earlier record to save. The test prevents the parser from creating an invalid dictionary entry before a genuine identifier exists.

### Why reset `current_parts`?

After saving the previous sequence, a new empty list must be created. Otherwise, fragments from the previous record would remain and become part of the next sequence.

### Why save once after the loop?

Records are saved when the next header arrives. The final record has no next header, so it must be stored after the loop ends. Forgetting this step is a common parsing error that silently loses the final record.

# Part 14 — Writing text files

Write mode `"w"` creates a file or replaces its existing contents.

In [7]:
with open("lecture07_summary.txt", "w") as output_file:
    output_file.write("Sequence summary\n")
    output_file.write("Number of records: " + str(len(sequences)) + "\n")

print("Summary file written.")

Summary file written.


`.write()` expects a string. `len(sequences)` produces an integer, so `str(...)` converts it before string concatenation.

Unlike `print()`, `.write()` does not automatically add a newline. The `\n` characters are included explicitly.

> **Important:** Opening an existing file with `"w"` immediately truncates it to zero length. Confirm the output path before running the cell.

Read the saved file back rather than assuming that writing succeeded.

In [8]:
with open("lecture07_summary.txt", "r") as output_file:
    saved_summary = output_file.read()

print(saved_summary)

Sequence summary
Number of records: 5



## Append mode

Append mode `"a"` adds text at the end of a file.

In [ ]:
with open("lecture07_summary.txt", "a") as output_file:
    output_file.write("Processing completed.\n")

Running an append cell repeatedly adds the same line repeatedly. Append mode is useful for logs, but repeated notebook execution must be considered.

For reproducible analysis outputs, write mode is often preferable because each run creates a fresh, complete result rather than accumulating old and new lines.

# Part 15 — Writing FASTA records

A FASTA writer must produce a header beginning with `>` followed by sequence text.

This example writes one sequence per line.

In [ ]:
with open("lecture07_uppercase_sequences.fasta", "w") as output_fasta:
    for sequence_id, sequence in sequences.items():
        output_fasta.write(">" + sequence_id + "\n")
        output_fasta.write(sequence + "\n")

print("FASTA file written.")

For each dictionary entry:

1. `">" + sequence_id + "\n"` constructs the header line;
2. `sequence + "\n"` constructs the sequence line;
3. each string is written to the open output file.

The dictionary stores identifiers but not the original descriptive header text. Therefore, this output contains only identifiers. Preserving descriptions would require storing them separately or storing richer records.

In [ ]:
with open("lecture07_uppercase_sequences.fasta", "r") as output_fasta:
    print(output_fasta.read())

# Part 23 — Guided exercises

## Exercise 1: inspect the FASTA file

Write code that:

1. opens `lecture07_sequences.fasta` safely;
2. counts all physical lines;
3. counts header lines;
4. counts non-empty sequence lines;
5. prints the three counts.

### Pseudocode

```text
set three counters to zero
open the FASTA file
for every line:
    increase the physical-line counter
    remove surrounding whitespace
    if the line begins with >:
        increase the header counter
    else if the line is not empty:
        increase the sequence-line counter
print all counters
```

In [ ]:
# Write Exercise 1 here.

### Exercise 1 — one possible solution

In [ ]:
physical_lines = 0
header_lines = 0
sequence_lines = 0

with open("lecture07_sequences.fasta", "r") as fasta_file:
    for line in fasta_file:
        physical_lines += 1
        clean_line = line.strip()

        if clean_line.startswith(">"):
            header_lines += 1
        elif clean_line != "":
            sequence_lines += 1

print("Physical lines:", physical_lines)
print("Header lines:", header_lines)
print("Sequence lines:", sequence_lines)

`+= 1` is shorthand for adding one to the existing value. The `elif` prevents header lines from also being counted as sequence lines.

## Exercise 2: longest sequence

Use the parsed dictionary to identify the longest sequence. Print its identifier and length.

### Pseudocode

```text
set longest identifier to nothing
set longest length to zero
for each identifier and sequence:
    calculate its length
    if the length is greater than the longest length:
        save this identifier
        save this length
print the saved identifier and length
```

In [ ]:
# Write Exercise 2 here.

### Exercise 2 — one possible solution

In [ ]:
longest_id = None
longest_length = 0

for sequence_id, sequence in loaded_sequences.items():
    sequence_length = len(sequence)

    if sequence_length > longest_length:
        longest_id = sequence_id
        longest_length = sequence_length

print("Longest identifier:", longest_id)
print("Longest length:", longest_length)

The stored values change only when a strictly longer sequence is found. If two sequences tie, this version retains the first one encountered. Tie handling should be changed if all longest records are required.

## Exercise 3: select sequences by length

Create a new dictionary containing sequences whose length is at least 35 nucleotides. Write the selected sequences to `lecture07_long_sequences.fasta`.

### Pseudocode

```text
create an empty dictionary
for each parsed sequence:
    if its length is at least 35:
        add it to the new dictionary
write the new dictionary as FASTA
reopen the output and inspect it
```

In [ ]:
# Write Exercise 3 here.

### Exercise 3 — one possible solution

In [ ]:
long_sequences = {}

for sequence_id, sequence in loaded_sequences.items():
    if len(sequence) >= 35:
        long_sequences[sequence_id] = sequence

write_fasta(
    long_sequences,
    "lecture07_long_sequences.fasta",
    line_width=60
)

with open("lecture07_long_sequences.fasta", "r") as check_file:
    print(check_file.read())

# Part 24 — Independent practice

Create a Python program that processes a supplied FASTA file and produces two outputs.

## Output 1: sequence report

The TSV report should contain:

- identifier;
- length;
- count of `A`;
- count of `C`;
- count of `G`;
- count of `T`;
- count of `N`;
- GC percentage;
- validation status.

## Output 2: filtered FASTA

Write only records that:

- contain no invalid symbols under the chosen alphabet;
- are at least 20 nucleotides long.

## Required safeguards

The program should:

1. preserve the input file;
2. use `with open(...)`;
3. give clear output filenames;
4. check that the input exists;
5. document whether `N` is allowed;
6. test at least two helper functions with `assert`;
7. reopen the outputs and verify their record or line counts.

Begin with pseudocode before writing Python.

In [ ]:
# Develop the independent practice program here.

# Lecture summary

This lecture developed a complete path from ordinary text files to reusable FASTA processing.

The main ideas are:

- a file stores information beyond the current Python session;
- paths tell Python where files are located;
- `with open(...)` manages file closing safely;
- `.read()`, `.readline()`, `.readlines()` and direct iteration serve different purposes;
- newline characters are part of text read from files;
- `.strip()` returns a new string;
- FASTA records begin with `>` headers and may contain several sequence lines;
- a multi-record parser must save the previous record and the final record;
- dictionaries connect sequence identifiers with sequences;
- normalization and validation are different operations;
- write mode can erase existing contents;
- outputs should be reopened and checked;
- small functions and tests make a workflow easier to understand and maintain.

Lecture 8 can build on this foundation by performing a larger sequence-analysis workflow or by introducing additional command-line processing, depending on the course plan.